# PhishGuard-ML: 03 - Model Training and Hyperparameter Tuning

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.model_selection import GridSearchCV

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import CategoricalNB

## 1. Load Data

In [2]:
X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv")['Result']
y_test = pd.read_csv("data/y_test.csv")['Result']

feature_importance = pd.read_csv("data/feature_importance.csv", index_col=0)
top_20_features = feature_importance.head(20).index.tolist()

# Let's train using all features for this demo, 
# but you can easily slice X_train[top_20_features] if needed.

## 2. Feature Mapping for Categorical Naïve Bayes

Categorical Naive Bayes requires non-negative integers. 
The dataset has values -1, 0, 1. We map them to 0, 1, 2.

In [3]:
def shift_features(X):
    # Mapping -1 -> 0, 0 -> 1, 1 -> 2
    return X + 1

## 3. Defining Pipelines and Grids

In [4]:
# Decision Tree Pipeline (No scaling needed)
dt_pipeline = Pipeline([
    ('clf', DecisionTreeClassifier(random_state=42))
])

dt_param_grid = {
    'clf__criterion': ['gini', 'entropy'],
    'clf__max_depth': [None, 10, 20, 30],
    'clf__min_samples_leaf': [1, 2, 4]
}

# KNN Pipeline (Requires scaling)
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', KNeighborsClassifier())
])

knn_param_grid = {
    'clf__n_neighbors': [3, 5, 7, 9, 11],
    'clf__weights': ['uniform', 'distance']
}

# SVM Pipeline (Requires scaling)
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(probability=True, random_state=42))
])

svm_param_grid = {
    'clf__kernel': ['linear', 'rbf'],
    'clf__C': [0.1, 1, 10],
    'clf__gamma': ['scale', 'auto']
}

# Categorical Naive Bayes Pipeline (Requires non-negative features, no scaling)
nb_pipeline = Pipeline([
    ('shifter', FunctionTransformer(shift_features)),
    ('clf', CategoricalNB())
])

nb_param_grid = {
    'clf__alpha': [0.1, 0.5, 1.0, 2.0]
}

## 4. Training and Tuning

In [5]:
models_and_grids = {
    "Decision Tree": (dt_pipeline, dt_param_grid),
    "KNN": (knn_pipeline, knn_param_grid),
    "SVM": (svm_pipeline, svm_param_grid),
    "Categorical Naive Bayes": (nb_pipeline, nb_param_grid)
}

best_estimators = {}

for name, (pipeline, param_grid) in models_and_grids.items():
    print(f"Training and tuning {name}...")
    # Use GridSearchCV with 5-fold stratified CV, optimizing for F1-score
    grid_search = GridSearchCV(
        pipeline, 
        param_grid, 
        cv=5, 
        scoring='f1',
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    
    print(f"Best parameters for {name}: {grid_search.best_params_}")
    print(f"Best cross-validation F1-score: {grid_search.best_score_:.4f}\n")
    
    best_estimators[name] = grid_search.best_estimator_

Training and tuning Decision Tree...
Best parameters for Decision Tree: {'clf__criterion': 'entropy', 'clf__max_depth': None, 'clf__min_samples_leaf': 1}
Best cross-validation F1-score: 0.9526

Training and tuning KNN...
Best parameters for KNN: {'clf__n_neighbors': 5, 'clf__weights': 'distance'}
Best cross-validation F1-score: 0.9474

Training and tuning SVM...
Best parameters for SVM: {'clf__C': 10, 'clf__gamma': 'scale', 'clf__kernel': 'rbf'}
Best cross-validation F1-score: 0.9584

Training and tuning Categorical Naive Bayes...
Best parameters for Categorical Naive Bayes: {'clf__alpha': 0.1}
Best cross-validation F1-score: 0.9188



## 5. Save the Models

In [6]:
os.makedirs("models", exist_ok=True)
for name, model in best_estimators.items():
    # Replace spaces with underscores for filename
    filename = name.replace(" ", "_").lower() + ".pkl"
    joblib.dump(model, f"models/{filename}")
    
print("All tuned models saved successfully.")

All tuned models saved successfully.
